# Blog figures — personal post (BLOG_DRAFT.md)

Every producible `[B#]` asset for the personal post. Runs on the workspace
against `$FM_BASE` (default `/mnt/user_storage/transaction-fm-v2`); every
loader skips loudly if its artifact is missing. Outputs land in the repo's `figures/` dir (commit them — the blogs embed
these paths directly); `$FM_FIG_OUT` overrides. PNG (200dpi) + SVG.

| cell | asset | source artifact |
|---|---|---|
| hero | **B1** two-panel dot+CI | `downstream/<sc>/bootstrap_ci.json` (2048 = 20-ep dir) + `<sc>_fulltest/` |
| arch | **B2** pipeline graphic (plain) | drawn here |
| fulltest | **B7** table-2 dot+CI | fulltest CI jsons + paired bootstrap json |
| reco | **B-RECO-FIG** slices bars | `full_nextmerchant/probe_metrics_v3.json` |
| burst | **B8** ledger + **B9** cumulative curve | `raw/full/transactions.parquet` (computed here) |
| canary | **B16** month canary + macro acc | `tensorboard/fm_<sc>_*` events |
| tables | **B10** velocity + final eyeball diff | control jsons + both CI sets |

**Cut, with reason:** B11 (surprise-vector anecdote) — no artifacts exist on
user_storage (searched `*surprise*`/`*probe*` 2026-07-14); the data was never
persisted from the research era. The draft section reads fine without it.

**Anyscale-post figures live in `blog_figures_anyscale.ipynb`.**


In [ ]:
import os, json, glob, subprocess, sys

for _pkg in ("plotly", "kaleido"):
    try:
        __import__(_pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd

BASE = os.environ.get("FM_BASE", "/mnt/user_storage/transaction-fm-v2")
_repo_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
FIG_OUT = os.environ.get("FM_FIG_OUT", os.path.join(_repo_root, "figures"))
os.makedirs(FIG_OUT, exist_ok=True)

# Anyscale brand (Notion Brand Hub go/brand): Anyscale Blue is the single accent;
# neutrals do the rest. Muted gray for context series, warm red only for failure.
ACCENT, MUTED, FAIL = "#0066FF", "#a8a6a0", "#c05a4e"
INK, INK2, GRID, SURFACE = "#000000", "#52514e", "#e6e0da", "#ffffff"
CTX_RAMP = {512: "#7fb0ff", 1024: "#0066FF", 2048: "#003c99"}  # ordinal blue ramp

pio.templates["anyscale"] = go.layout.Template(layout=dict(
    font=dict(family="Archivo, Inter, Helvetica, Arial", size=13, color=INK2),
    title=dict(font=dict(size=17, color=INK), x=0.02, xanchor="left"),
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    colorway=[ACCENT, MUTED, FAIL, "#4b34b3", "#00a2e9"],
    xaxis=dict(gridcolor=GRID, zeroline=False),
    yaxis=dict(gridcolor=GRID, zeroline=False),
    margin=dict(l=70, r=40, t=80, b=70),
    legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right"),
))
pio.templates.default = "anyscale"

def psave(fig, name, w=1250, h=540):
    fig.write_image(f"{FIG_OUT}/{name}.png", width=w, height=h, scale=2)
    fig.write_image(f"{FIG_OUT}/{name}.svg", width=w, height=h)
    print(f"[saved] {FIG_OUT}/{name}.png + .svg")

def load_results(path):
    if not os.path.exists(path):
        print(f"[SKIP] missing {path}")
        return None
    return json.load(open(path))["results"]

def add_ci_dots(fig, rows, col=1):
    """Dot + 95%-CI whiskers for (label, result-dict, is_ours) rows: whiskers wear the
    point color, accent points get their value annotated above the whisker top."""
    rows = [(l, r, o) for l, r, o in rows if r is not None]
    kw = {"row": 1, "col": col} if getattr(fig, "_grid_ref", None) else {}
    fig.update_xaxes(categoryorder="array", categoryarray=[l for l, _, _ in rows],
                     tickangle=22, **kw)
    for ours in (False, True):
        sub = [(l, r) for l, r, o in rows if o == ours]
        if not sub:
            continue
        c = ACCENT if ours else MUTED
        fig.add_trace(go.Scatter(
            x=[l for l, _ in sub], y=[r["ap"] for _, r in sub], mode="markers",
            marker=dict(size=10, color=c),
            error_y=dict(type="data", color=c, thickness=1.6, width=5,
                         array=[r["ap_ci95"][1] - r["ap"] for _, r in sub],
                         arrayminus=[r["ap"] - r["ap_ci95"][0] for _, r in sub]),
            cliponaxis=False, showlegend=False,
            hovertemplate="%{x}: AP %{y:.4f}<extra></extra>"), **kw)
        if ours:
            for l, r in sub:
                fig.add_annotation(x=l, y=r["ap_ci95"][1], text=f'{r["ap"]:.3f}',
                                   yanchor="bottom", yshift=3, showarrow=False,
                                   font=dict(size=12, color=INK), **kw)

ci100k = {sc: load_results(f"{BASE}/downstream/{sc}/bootstrap_ci.json") for sc, _ in [("full",512),("xl",1024),("xxl",2048)]}
# Table-1 2048 row = the 20-epoch eval (same training budget as 512/1024);
# downstream/xxl holds the 40-ep continuation rerun (table 2's model).
ci100k["xxl"] = (load_results(f"{BASE}/downstream/xxl_old_1783532341/bootstrap_ci.json")
                 or ci100k["xxl"])
cifull = {sc: load_results(f"{BASE}/downstream/{sc}_fulltest/bootstrap_ci.json") for sc, _ in [("full",512),("xl",1024),("xxl",2048)]}
SCALES = [("full", 512), ("xl", 1024), ("xxl", 2048)]
NV_BASELINE, NV_FUSION = 0.1238, 0.1755


## B1 — hero: two panels, dot + 95% CI

Left = their 100k protocol (the only NVIDIA-comparable set), with their two
published numbers as reference lines. Right = full test period. Shared y-axis;
the caption carries the never-compare-across-panels warning. Our
`embed_xgb` rows wear the accent; every other readout is context.

In [ ]:
if all(ci100k[sc] for sc, _ in SCALES) and all(cifull[sc] for sc, _ in SCALES):
    left = [
        ("raw-13 baseline", ci100k["full"].get("baseline"),        False),
        ("PCA64 · 512",     ci100k["full"].get("embed_pca64_xgb"), False),
        ("PCA64 · 1024",    ci100k["xl"].get("embed_pca64_xgb"),   False),
        ("linear · 512",    ci100k["full"].get("embed_logistic"),  False),
        ("XGB · 512",       ci100k["full"].get("embed_xgb"),       True),
        ("XGB · 1024",      ci100k["xl"].get("embed_xgb"),         True),
        ("XGB · 2048 (20ep)", ci100k["xxl"].get("embed_xgb"),      True),
    ]
    right = [
        ("raw-13 baseline",   cifull["full"].get("baseline"),  False),
        ("XGB · 512",         cifull["full"].get("embed_xgb"), True),
        ("XGB · 1024",        cifull["xl"].get("embed_xgb"),   True),
        ("XGB · 2048 (40ep)", cifull["xxl"].get("embed_xgb"),  True),
    ]
    fig = make_subplots(rows=1, cols=2, shared_yaxes=True, column_widths=[0.62, 0.38],
                        horizontal_spacing=0.045,
                        subplot_titles=("Their protocol — 100k test (112 frauds)",
                                        "Full test period — 2.44M rows (2,724 frauds)"))
    for col, rows in ((1, left), (2, right)):
        add_ci_dots(fig, rows, col=col)
    for y, lab, c in ((NV_FUSION, f"NVIDIA published fusion {NV_FUSION}", INK2),
                      (NV_BASELINE, f"NVIDIA published baseline {NV_BASELINE}", MUTED)):
        fig.add_hline(y=y, line=dict(color=c, width=1.1, dash="dash"), row=1, col=1)
        fig.add_annotation(text=lab, xref="x domain", yref="y", x=0.98, y=y,
                           xanchor="right", yanchor="bottom", showarrow=False,
                           font=dict(size=11, color=c), bgcolor=SURFACE, row=1, col=1)
    fig.update_yaxes(title_text="Average precision (test)", row=1, col=1)
    fig.update_xaxes(tickangle=22)
    fig.update_layout(
        title="The embedding alone clears their published fusion headline —<br>"
              "and the lift is CI-separated on the full test period",
        margin=dict(t=110, b=110, r=70), height=580)
    fig.add_annotation(text="Dots = point AP, whiskers = bootstrap 95% CI. "
                            "The two panels are different eval sets — never compare numbers across them.",
                       xref="paper", yref="paper", x=0.5, y=-0.24, showarrow=False,
                       font=dict(size=12, color=INK2))
    psave(fig, "b1_hero", w=1250, h=560)
    fig.show()
else:
    print("[SKIP] B1 — need bootstrap_ci.json for all scales (100k + fulltest)")


## B2 — pipeline architecture

Now a **D2 diagram** (`figures/src/architecture.d2`, checked in — diagrams-as-code, house-style
pastel palette). This cell re-renders it if the `d2` binary is available; otherwise the
committed SVG/PNG are canonical. The Anyscale-annotated variant lives in the same source.


In [ ]:
import shutil, subprocess
d2src = os.path.join(_repo_root, "figures", "src", "architecture.d2")
if shutil.which("d2") and os.path.exists(d2src):
    subprocess.run(["d2", "--layout", "elk", "--pad", "24", d2src, f"{FIG_OUT}/b2_architecture.svg"], check=True)
    if shutil.which("rsvg-convert"):
        subprocess.run(["rsvg-convert", "-w", "2400", f"{FIG_OUT}/b2_architecture.svg",
                        "-o", f"{FIG_OUT}/b2_architecture.png"], check=True)
    print(f"[saved] {FIG_OUT}/b2_architecture.svg (+png if rsvg-convert present)")
else:
    print(f"[SKIP] d2 render — binary or {d2src} missing; committed figures/b2_architecture.* are canonical")


## B7 — table 2 as a dot-and-CI plot

Every fulltest row; the raw baseline's CI drawn as a reference band so
"CI-separated" is visible rather than asserted. Paired-ordering stats from
`paired_bootstrap_embed_xgb.json` go in the caption.

In [ ]:
pb_path = f"{BASE}/downstream/paired_bootstrap_embed_xgb.json"
pb = json.load(open(pb_path)) if os.path.exists(pb_path) else None

if all(cifull[sc] for sc, _ in SCALES):
    rows = [
        ("PCA64 · 512",       cifull["full"].get("embed_pca64_xgb"), False),
        ("PCA64 · 1024",      cifull["xl"].get("embed_pca64_xgb"),   False),
        ("linear · 512",      cifull["full"].get("embed_logistic"),  False),
        ("linear · 1024",     cifull["xl"].get("embed_logistic"),    False),
        ("XGB · 512",         cifull["full"].get("embed_xgb"),       True),
        ("XGB · 1024",        cifull["xl"].get("embed_xgb"),         True),
        ("XGB · 2048 (40ep)", cifull["xxl"].get("embed_xgb"),        True),
    ]
    base_r = cifull["full"]["baseline"]
    fig = go.Figure()
    fig.add_hrect(y0=base_r["ap_ci95"][0], y1=base_r["ap_ci95"][1],
                  fillcolor=GRID, opacity=0.5, line_width=0)
    fig.add_hline(y=base_r["ap"], line=dict(color=INK2, width=1))
    fig.add_annotation(text=f'raw-13 baseline {base_r["ap"]:.3f} (95% CI band)',
                       xref="x domain", x=0.985, y=base_r["ap"], xanchor="right",
                       yanchor="bottom", showarrow=False, font=dict(size=11, color=INK2))
    add_ci_dots(fig, rows)
    cap = "Dots = point AP, whiskers = bootstrap 95% CI."
    if pb:
        ctx = {"full_fulltest": "512", "xl_fulltest": "1024", "xxl_fulltest": "2048"}
        pieces = [f'{ctx.get(a.strip(), a)}−{ctx.get(b.strip(), b)}: {v["mean_diff"]:+.3f} '
                  f'[{v["ci95"][0]:+.3f}, {v["ci95"][1]:+.3f}] P={v["p_a_gt_b"]:.3f}'
                  for k, v in pb["pairs"].items() for a, _, b in [k.partition(" - ")]]
        cap += "<br>Paired bootstrap (same resampled rows): " + " · ".join(pieces)
    fig.update_layout(title="Full test period (2.44M rows): all models on identical rows",
                      yaxis_title="Average precision (test)", height=560,
                      margin=dict(t=80, b=130, r=70))
    fig.add_annotation(text=cap, xref="paper", yref="paper", x=0.5, y=-0.28,
                       showarrow=False, font=dict(size=11, color=INK2))
    psave(fig, "b7_fulltest", w=1150, h=560)
    fig.show()
else:
    print("[SKIP] B7 — need fulltest bootstrap_ci.json for all scales")


## B-RECO-FIG — where the FM earns its budget

Grouped bars, HR@10: overall (honest memorization floor vs the hybrid), the
next-merchant∉top-10 slice (memorization structurally 0), and never-seen
merchants (any history method structurally 0). Accent = FM-powered method
(hybrid overall, full-vocab MLP on the slices — labeled per group).

In [ ]:
v3p = f"{BASE}/downstream/full_nextmerchant/probe_metrics_v3.json"
if os.path.exists(v3p):
    v3 = json.load(open(v3p))
    sl = v3["slices"]
    groups = [
        ("Overall (400k events)<br><i>hybrid: 0.1·MLP + 0.9·freq</i>",
         v3["readouts"]["naive_count"]["hr@10"], v3["readouts"]["alpha_blend"]["hr@10"]),
        (f"Next merchant NOT in card top-10 ({sl['share_not_in_top10']:.0%})<br><i>full-vocab MLP</i>",
         sl["not_in_top10"]["naive_count"], sl["not_in_top10"]["mlp_solo_token_space"]),
        (f"Never-seen merchant ({sl['share_never_seen_or_beyond_cap']:.0%})<br><i>full-vocab MLP</i>",
         sl["never_seen_or_beyond_cap"]["naive_count"],
         sl["never_seen_or_beyond_cap"]["mlp_solo_token_space"]),
    ]
    names = [g[0] for g in groups]
    fig = go.Figure([
        go.Bar(x=names, y=[g[1] for g in groups], name="memorization baseline (top-10 history)",
               marker_color=MUTED, text=[f"{g[1]:.3f}" if g[1] else "0" for g in groups],
               textposition="outside"),
        go.Bar(x=names, y=[g[2] for g in groups], name="FM-powered",
               marker_color=ACCENT, text=[f"{g[2]:.3f}" for g in groups],
               textposition="outside"),
    ])
    fig.update_layout(barmode="group", bargap=0.35, bargroupgap=0.08,
                      yaxis_title="HR@10", height=520,
                      title="The recommender beats memorization overall — and covers the slices<br>"
                            "memorization is structurally blind to")
    psave(fig, "b_reco_slices", w=1100, h=520)
    fig.show()
else:
    print(f"[SKIP] B-RECO-FIG — missing {v3p}")


## B8 + B9 — the burst stat, computed and ledgered

The draft's "90% of test frauds have a prior same-card fraud within the
preceding 512 transactions vs 7.3% of normals" currently lives in campaign
memory, not an artifact. This computes it from the raw parquet (distance in
same-card transactions to the most recent *strictly prior* fraud), writes
`burst_stats.json` as the citable ledger, and draws the histogram with the
context ceilings (~315 = their token budget, 512/1024/2048 = ours) marked.

In [ ]:
import pyarrow.dataset as pads

raw_p = f"{BASE}/raw/full/transactions.parquet"
splits_p = f"{BASE}/raw/full/splits.json"
if os.path.exists(splits_p):
    splits = json.load(open(splits_p))
    df = (pads.dataset(raw_p, format="parquet")
              .to_table(columns=["card_id", "timestamp", "is_fraud"]).to_pandas())
    df = df.sort_values(["card_id", "timestamp"], kind="stable").reset_index(drop=True)
    is_fraud = df.is_fraud.astype(bool)
    print(f"{len(df):,} transactions, {is_fraud.sum():,} frauds ({is_fraud.mean():.4%})")

    pos = df.groupby("card_id", sort=False).cumcount()
    fraud_pos = pos.where(is_fraud)
    # last fraud at-or-before each row, then shift within card -> strictly prior
    prior = fraud_pos.groupby(df.card_id, sort=False).ffill()
    prior = prior.groupby(df.card_id, sort=False).shift(1)
    dist = pos - prior  # NaN = no prior same-card fraud

    # unit-safe test mask: compare as pandas timestamps, never raw int64
    ts = pd.to_datetime(df["timestamp"])
    cut = pd.to_datetime(splits["val_end"])
    test = ts > cut
    n_test, n_test_fraud = int(test.sum()), int(is_fraud[test].sum())
    print(f"test period (> {cut}): {n_test:,} rows, {n_test_fraud:,} frauds")
    assert n_test > 0 and n_test_fraud > 0, "test mask is empty — check splits val_end vs timestamp dtype"

    stats = {"definition": "distance in same-card transactions to most recent "
                           "strictly-prior fraud; test-period rows (ts > val_end)",
             "n_test": n_test, "n_test_fraud": n_test_fraud}
    groups = {}
    for grp_name, mask in (("fraud", test & is_fraud), ("normal", test & ~is_fraud)):
        d_grp = dist[mask]
        groups[grp_name] = d_grp
        s = {"n": int(mask.sum()), "share_no_prior_fraud": float(d_grp.isna().mean())}
        for k in (315, 512, 1024, 2048):
            s[f"share_within_{k}"] = float((d_grp <= k).mean())
        stats[grp_name] = s
        print(f"  {grp_name}: n={s['n']:,}  within 512 = {s['share_within_512']:.1%}  "
              f"no prior fraud = {s['share_no_prior_fraud']:.1%}")
    with open(f"{FIG_OUT}/burst_stats.json", "w") as f:
        json.dump(stats, f, indent=2)
    print(f"[saved] {FIG_OUT}/burst_stats.json  <- cite these exact numbers in B8")

    # Cumulative form: share of the group whose most recent same-card fraud is
    # within x transactions — the 90%-vs-7.3% claim is read straight off x=512.
    fig = go.Figure()
    xmax = max(float(dist.max()), 4096.0)
    for name, key, color in (("fraud txns", "fraud", ACCENT), ("normal txns", "normal", MUTED)):
        d_grp = groups[key].dropna().sort_values().to_numpy()
        n_grp = stats[key]["n"]
        xs = np.concatenate([[1.0], d_grp.clip(min=1.0), [xmax]])
        ys = np.concatenate([[0.0], (np.arange(len(d_grp)) + 1) / n_grp,
                             [len(d_grp) / n_grp]])
        fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line_shape="hv",
                                 line=dict(color=color, width=2.6), showlegend=False,
                                 hovertemplate=name + ": %{y:.1%} within %{x:.0f}<extra></extra>"))
        at512 = stats[key]["share_within_512"]
        lx, ly = (0.32, 0.72) if key == "fraud" else (0.32, 0.16)
        fig.add_annotation(text=f"<b>{name}: {at512:.0%} within 512</b>", xref="paper",
                           yref="paper", x=lx, y=ly, showarrow=False,
                           font=dict(size=14, color=color))
    for k, lab in ((315, "~315 (their budget)"), (512, "512"), (1024, "1024"), (2048, "2048")):
        fig.add_vline(x=k, line=dict(color=GRID, width=1, dash="dash"))
        fig.add_annotation(text=lab, x=np.log10(k), yref="paper", y=1.03, showarrow=False,
                           xanchor="right" if k == 315 else "center",
                           font=dict(size=11, color=INK2))
    fig.update_xaxes(type="log", range=[0, np.log10(xmax)], dtick=1,
                     title="same-card transactions since previous fraud (log scale)")
    fig.update_yaxes(range=[0, 1.02], tickformat=".0%",
                     title="share of group with a prior fraud this recent")
    fig.update_layout(title="Fraud is bursty: the signal sits within the context window",
                      height=540, margin=dict(t=90))
    psave(fig, "b9_burst", w=1150, h=540)
    fig.show()
else:
    print(f"[SKIP] B8/B9 — missing {splits_p}")


## B16 — the month canary + macro accuracy

`field_ce/month` per optimizer step (same optimizer work = fair x-axis across
context lengths) and `epoch/acc_macro` per epoch, all three scales on the
validated ordinal blue ramp. The 2048 run's 20→40 continuation appears
naturally (its resumed run carries both event streams).

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

def scalars(scale, tag):
    pts = []
    for run in sorted(glob.glob(f"{BASE}/tensorboard/fm_{scale}_*")):
        acc = EventAccumulator(run); acc.Reload()
        if tag in acc.Tags()["scalars"]:
            pts += [(e.step, e.value) for e in acc.Scalars(tag)]
    return sorted(set(pts)) or None

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.07,
                    subplot_titles=("month canary — field_ce/month (lower = learned)",
                                    "per-field macro accuracy by epoch"))
found = False
for scale, ctx in SCALES:
    for col, tag in ((1, "field_ce/month"), (2, "epoch/acc_macro")):
        pts = scalars(scale, tag)
        if pts:
            found = True
            xs, ys = zip(*pts)
            fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines",
                                     line=dict(color=CTX_RAMP[ctx], width=2.4),
                                     name=f"{ctx} ctx", legendgroup=ctx,
                                     showlegend=(col == 1)), row=1, col=col)
if found:
    fig.update_xaxes(title_text="optimizer step", row=1, col=1)
    fig.update_xaxes(title_text="epoch", row=1, col=2)
    fig.update_yaxes(title_text="cross-entropy", row=1, col=1)
    fig.update_yaxes(title_text="accuracy", row=1, col=2)
    fig.update_layout(title="Longer context = fewer windows per epoch: the 512 run hits its month<br>"
                            "phase transition; the longer runs grind toward theirs",
                      height=520, margin=dict(t=110))
    psave(fig, "b16_canary", w=1250, h=520)
    fig.show()
else:
    print(f"[SKIP] B16 — no event files under {BASE}/tensorboard/fm_*")


## B10 + final eyeball diff

The velocity-control numbers for the B10 table, the shuffled-label control,
and both CI tables printed once more — diff these against the draft's inline
tables *before* exporting anything above, so figures and prose can't drift.

In [ ]:
for p, label in (
        (f"{BASE}/downstream/full_velocity/velocity_metrics.json", "B10 velocity control"),
        (f"{BASE}/downstream/full_probe/probe_metrics_shuffled_seed0.json",
         "shuffled-label control")):
    if os.path.exists(p):
        print(f"== {label}: {p}")
        print(json.dumps(json.load(open(p)), indent=2)[:2000])
    else:
        print(f"[SKIP] {label} — missing {p}")

for name, src in (("TABLE 1 (100k)", ci100k), ("TABLE 2 (fulltest)", cifull)):
    rows = []
    for sc, ctx in SCALES:
        if src[sc]:
            for m, r in src[sc].items():
                rows.append({"model": m, "ctx": ctx, "ap": round(r["ap"], 4),
                             "ci": [round(x, 3) for x in r["ap_ci95"]],
                             "P(>fusion)": round(r.get("p_beats_nvidia_fusion",
                                                       float("nan")), 3)})
    if rows:
        print(f"\n== {name} — diff against BLOG_DRAFT.md")
        print(pd.DataFrame(rows).to_string(index=False))

---
**Produced here:** `b1_hero`, `b2_architecture`, `b7_fulltest`, `b_reco_slices`,
`b9_burst` (+ `burst_stats.json`, the B8 ledger), `b16_canary` — PNG+SVG each.

**Still manual:** Zach sign-offs + links, venue/CTA decisions. B11 cut (no data).
**Anyscale post:** `blog_figures_anyscale.ipynb`.
